# Phase 2.5: Best Behavioral Feature Combination

This focused experiment combines the two strongest existing behavioral groups: historical amounts and recency. It introduces no new features and fits one new fixed-parameter LightGBM model. The other three comparison rows are the verified results from the executed Phase 2 notebook.

In [1]:
import gc
import sys
import time
from pathlib import Path

import lightgbm as lgb
import mlflow
import pandas as pd
from IPython.display import display
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from fraud_detection.data import (
    load_train_identity,
    load_train_transaction,
    merge_train_data,
    split_data_chronologically,
)
from fraud_detection.features import BehavioralFeatureEngineer, Phase1FeatureEngineer
from fraud_detection.models import TrainingOnlyPreprocessor, evaluate_probabilities

experiment_started = time.perf_counter()

In [2]:
raw_dir = PROJECT_ROOT / "data" / "raw"
transaction = load_train_transaction(raw_dir)
identity = load_train_identity(raw_dir)
merged = merge_train_data(transaction, identity)
split = split_data_chronologically(merged)
train = split.train
validation = split.validation
del transaction, identity, merged
gc.collect()

predictor_columns = [
    column for column in train.columns if column not in ("isFraud", "TransactionID")
]
train_predictors = train.loc[:, predictor_columns]
validation_predictors = validation.loc[:, predictor_columns]
y_train = train["isFraud"].astype("int8")
y_validation = validation["isFraud"].astype("int8")

frequency_engineer = Phase1FeatureEngineer(
    include_amount_time=False,
    include_missing_indicators=False,
    include_frequency_encodings=True,
).fit(train_predictors)
frequency_train = frequency_engineer.transform(train_predictors)
frequency_validation = frequency_engineer.transform(validation_predictors)

behavior_source_columns = [
    column
    for column in (
        "TransactionID",
        "TransactionDT",
        "TransactionAmt",
        "card1",
        "addr1",
        "DeviceInfo",
        "P_emaildomain",
    )
    if column in train.columns
]
behavioral_engineer = BehavioralFeatureEngineer(
    include_counts=False,
    include_amounts=True,
    include_recency=True,
    include_velocity=False,
)
behavioral_train = behavioral_engineer.fit_transform_features(train.loc[:, behavior_source_columns])
behavioral_validation = behavioral_engineer.transform_features(
    validation.loc[:, behavior_source_columns], update_state=True
)
selected_behavioral_features = behavioral_engineer.generated_feature_names_
combined_train = pd.concat(
    [frequency_train, behavioral_train.loc[:, list(selected_behavioral_features)]], axis=1
)
combined_validation = pd.concat(
    [
        frequency_validation,
        behavioral_validation.loc[:, list(selected_behavioral_features)],
    ],
    axis=1,
)

print(f"Train rows: {len(train):,}")
print(f"Validation rows: {len(validation):,}")
print(f"Frequency feature count: {frequency_train.shape[1]}")
print(f"Selected behavioral features: {len(selected_behavioral_features)}")
print(f"Combined feature count: {combined_train.shape[1]}")
print("Reserved final split status: not assigned or accessed")
selected_behavioral_features

Train rows: 413,378
Validation rows: 88,581
Frequency feature count: 438
Selected behavioral features: 7
Combined feature count: 445
Reserved final split status: not assigned or accessed


('card1_previous_avg_amount',
 'card1_previous_max_amount',
 'amount_vs_card1_previous_avg',
 'addr1_previous_avg_amount',
 'amount_vs_addr1_previous_avg',
 'seconds_since_card1_previous_transaction',
 'seconds_since_device_previous_transaction')

In [3]:
lightgbm_parameters = dict(
    learning_rate=0.1,
    n_estimators=400,
    num_leaves=63,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)
preprocessor = TrainingOnlyPreprocessor()
x_train = preprocessor.fit_transform(combined_train)
x_validation = preprocessor.transform(combined_validation)
scaler = StandardScaler(copy=False)
x_train = scaler.fit_transform(x_train)
x_validation = scaler.transform(x_validation)

tracking_uri = f"sqlite:///{(PROJECT_ROOT / 'model_comparison.db').resolve().as_posix()}"
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("fraud-detection-behavioral-features")
with mlflow.start_run(run_name="frequency_plus_amounts_plus_recency") as run:
    model = lgb.LGBMClassifier(**lightgbm_parameters)
    model.fit(x_train, y_train)
    probabilities = model.predict_proba(x_validation)[:, 1]
    combined_metrics = evaluate_probabilities(y_validation, probabilities)
    mlflow.set_tags(
        {
            "stage": "behavioral-feature-engineering",
            "dataset": "ieee-cis",
            "split": "temporal-validation",
        }
    )
    mlflow.log_params(
        {
            "feature_set_name": "frequency_plus_amounts_plus_recency",
            "feature_count": x_train.shape[1],
            "learning_rate": lightgbm_parameters["learning_rate"],
            "n_estimators": lightgbm_parameters["n_estimators"],
            "num_leaves": lightgbm_parameters["num_leaves"],
        }
    )
    mlflow.log_metrics(
        {
            "pr_auc": combined_metrics.pr_auc,
            "roc_auc": combined_metrics.roc_auc,
            "precision_at_0_50": combined_metrics.precision,
            "recall_at_0_50": combined_metrics.recall,
            "f1_at_0_50": combined_metrics.f1,
        }
    )
    combined_run_id = run.info.run_id

print(f"Combined validation PR-AUC: {combined_metrics.pr_auc:.6f}")
print(f"MLflow run ID: {combined_run_id}")

Combined validation PR-AUC: 0.589112
MLflow run ID: cb3f6478222e49afb72df8b1155453a6


In [4]:
comparison = pd.DataFrame(
    [
        {
            "Feature Set": "frequency_only",
            "Feature Count": 438,
            "PR-AUC": 0.589631,
            "ROC-AUC": 0.921839,
            "Precision": 0.577205,
            "Recall": 0.544379,
            "F1": 0.560311,
        },
        {
            "Feature Set": "frequency_plus_historical_amounts",
            "Feature Count": 443,
            "PR-AUC": 0.593938,
            "ROC-AUC": 0.919463,
            "Precision": 0.573609,
            "Recall": 0.555884,
            "F1": 0.564608,
        },
        {
            "Feature Set": "frequency_plus_recency",
            "Feature Count": 440,
            "PR-AUC": 0.593936,
            "ROC-AUC": 0.924308,
            "Precision": 0.534618,
            "Recall": 0.583826,
            "F1": 0.558140,
        },
        {
            "Feature Set": "frequency_plus_amounts_plus_recency",
            "Feature Count": x_train.shape[1],
            "PR-AUC": combined_metrics.pr_auc,
            "ROC-AUC": combined_metrics.roc_auc,
            "Precision": combined_metrics.precision,
            "Recall": combined_metrics.recall,
            "F1": combined_metrics.f1,
        },
    ]
).sort_values("PR-AUC", ascending=False, ignore_index=True)
best = comparison.iloc[0]
improvement = combined_metrics.pr_auc - 0.593938
relative_improvement = improvement / 0.593938
display(comparison)


def markdown_table(frame):
    rows = [
        "| Feature set | Features | PR-AUC | ROC-AUC | Precision | Recall | F1 |",
        "| --- | ---: | ---: | ---: | ---: | ---: | ---: |",
    ]
    for _, row in frame.iterrows():
        rows.append(
            f"| {row['Feature Set']} | {int(row['Feature Count'])} | "
            f"{row['PR-AUC']:.6f} | {row['ROC-AUC']:.6f} | "
            f"{row['Precision']:.6f} | {row['Recall']:.6f} | {row['F1']:.6f} |"
        )
    return "\n".join(rows)


outcome = "improved" if improvement > 0 else "did not improve"
section = f"""
## Best feature combination

This focused Phase 2.5 experiment reused the fixed LightGBM parameters and compared exactly
the two strongest existing behavioral groups, separately and together. The first three rows
are verified Phase 2 results; only the combined configuration required a new fit.

{markdown_table(comparison)}

- Winner: **{best["Feature Set"]}** with **{best["PR-AUC"]:.6f}** PR-AUC and
  **{int(best["Feature Count"])}** features.
- The amounts-plus-recency combination **{outcome}** on the previous best of `0.593938`.
- Combined absolute change: **{improvement:+.6f}**.
- Combined relative change: **{relative_improvement:+.2%}**.
- Combined MLflow run: `{combined_run_id}`.
- Experiment runtime: **{time.perf_counter() - experiment_started:.1f} seconds**.

Behavioral state was fitted from TRAIN without labels. VALIDATION features used TRAIN history
and earlier validation transactions only. The reserved final split was not accessed.
"""
report_path = PROJECT_ROOT / "reports" / "behavioral_features_summary.md"
existing_report = report_path.read_text(encoding="utf-8")
marker = "\n## Best feature combination\n"
base_report = existing_report.split(marker, maxsplit=1)[0].rstrip()
report_path.write_text(base_report + "\n" + section, encoding="utf-8")
print(f"Winner: {best['Feature Set']} ({best['PR-AUC']:.6f})")
print(f"Combined absolute PR-AUC change: {improvement:+.6f}")
print(f"Combined relative PR-AUC change: {relative_improvement:+.2%}")

,Feature Set,Feature Count,PR-AUC,ROC-AUC,Precision,Recall,F1
0,frequency_plus_historical_amounts,443,0.593938,0.919463,0.573609,0.555884,0.564608
1,frequency_plus_recency,440,0.593936,0.924308,0.534618,0.583826,0.558140
2,frequency_only,438,0.589631,0.921839,0.577205,0.544379,0.560311
3,frequency_plus_amounts_plus_recency,445,0.589112,0.921464,0.556792,0.559172,0.557979


Winner: frequency_plus_historical_amounts (0.593938)
Combined absolute PR-AUC change: -0.004826
Combined relative PR-AUC change: -0.81%


## Scope boundary

This experiment stops after testing the existing historical-amount and recency combination. It adds no features, tuning, threshold optimization, graph analysis, explanation, or production work.